## Phase 4: Production Optimization & Security

This notebook prepares the final fraud monitoring dataset from the Gold layer.

Features implemented:

• Broadcast join optimization

• Latency tracking

• Strict PII masking

#### Load Gold

In [0]:
alerts = spark.table("fact_fraud_alerts")
risk = spark.table("dim_account_risk_score")

#### ->BroadCast Join

In [0]:
from pyspark.sql.functions import broadcast

final = alerts.join(
broadcast(risk),
"account_id"
)

#### ->PII Masking

In [0]:
from pyspark.sql.functions import sha2, col

final = final.withColumn(
"account_id_masked",
sha2(col("account_id").cast("string"),256)
)

#### ->Latency Monitoring

In [0]:
from pyspark.sql.functions import current_timestamp

final = final.withColumn(
"processing_time",
current_timestamp()
)

In [0]:
final.write.format("delta") \
.mode("overwrite") \
.saveAsTable("production_fraud_monitoring")